# Experiment 1: Syndrome Observability (Theorem 6.1) — DDPM 버전

## 목표
TOY 실험(Low-Rank Gaussian)에서 검증한 Theorem 6.1을
**실제 DDPM 환경에서도 동일하게 성립하는지** 확인한다.

## 핵심 변경사항 (TOY → DDPM)
| | TOY | DDPM |
|---|---|---|
| Posterior mean | Closed-form 공식 | Tweedie + UNet |
| Normal/Tangent basis | `model.U` (analytical) | **Stanczuk SVD** (근사) |
| 데이터 | Low-Rank Gaussian 샘플 | CIFAR-10 체크포인트 |

## 실험 흐름
```
① 모델 로드
② x_t 샘플 생성 (forward diffuse)
③ Stanczuk SVD → normal/tangent basis 추출  ← model.U 대체
④ 두 방향으로 perturbation 주입
⑤ syndrome 계산
⑥ 그래프: TOY 결과와 비교
```

## Cell 1: 환경 설정 및 import

In [ ]:
import sys
sys.path.append('./mainddpm')
sys.path.append('.')

import os
import yaml
import argparse
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# 재현성
SEED = 1234 + 9
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {DEVICE}')
print(f'PyTorch 버전: {torch.__version__}')

## Cell 2: Config 로드 및 모델 불러오기

기존 repo의 `runner.creat_model()`을 **그대로** 사용한다.  
diffusers 같은 외부 라이브러리 필요 없음.

In [ ]:
# ── config 로드 ──────────────────────────────────────────────────
CONFIG_PATH = './mainddpm/configs/cifar10.yml'

def dict2namespace(config):
    namespace = argparse.Namespace()
    for key, value in config.items():
        if isinstance(value, dict):
            new_value = dict2namespace(value)
        else:
            new_value = value
        setattr(namespace, key, new_value)
    return namespace

with open(CONFIG_PATH, 'r') as f:
    config_dict = yaml.safe_load(f)
config = dict2namespace(config_dict)

# ── args 설정 (ddim_cifar_siec.py와 동일한 방식) ─────────────────
args = argparse.Namespace(
    config       = CONFIG_PATH,
    seed         = SEED,
    device       = DEVICE,
    use_pretrained = True,
    timesteps    = 100,       # DDIM steps
    eta          = 0.0,
    skip_type    = 'quad',
    sample_type  = 'generalized',
    ni           = True,
    exp          = 'exp1',
    image_folder = './exp1_results',
    select_step  = None,
    select_depth = None,
    sequence     = False,
)

# ── 모델 로드 ─────────────────────────────────────────────────────
from ddpm.runners.diffusion import Diffusion

runner = Diffusion(args, config)
model  = runner.creat_model()   # ← 사전학습 체크포인트 자동 로드
model  = model.to(DEVICE)
model.eval()

# ── noise schedule (betas) 가져오기 ──────────────────────────────
betas = runner.betas.to(DEVICE)           # shape: [1000]
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)  # alpha_bar

# ── timestep 시퀀스 가져오기 ──────────────────────────────────────
seq, _ = runner.obtain_generator_para()   # DDIM skip 시퀀스
seq = list(seq)

print(f'모델 로드 완료')
print(f'betas shape    : {betas.shape}')
print(f'DDIM seq (앞5) : {seq[:5]}')
print(f'DDIM seq (뒤5) : {seq[-5:]}')

## Cell 3: 핵심 함수 정의

TOY 코드의 각 함수를 DDPM 버전으로 교체한다.

```
TOY                            DDPM
──────────────────────────     ──────────────────────────────
model.posterior_mean()    →    tweedie_x0()
model.forward()           →    forward_diffuse()
decompose_error(err, U)   →    decompose_error(err, N, T)
ddim_step()               →    ddim_step()
model.U                   →    get_stanczuk_basis()   ← Stanczuk SVD
```

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 함수 1: Forward diffuse  (TOY의 model.forward() 대체)
#   x_0 → x_t  (노이즈 추가)
# ══════════════════════════════════════════════════════════════════
def forward_diffuse(x0, t_idx):
    """
    x0    : 깨끗한 이미지 [B, C, H, W]
    t_idx : timestep 인덱스 (0~999)
    return: x_t (노이즈 낀 이미지)
    """
    alpha_bar = alphas_cumprod[t_idx]
    eps = torch.randn_like(x0)
    x_t = torch.sqrt(alpha_bar) * x0 + torch.sqrt(1 - alpha_bar) * eps
    return x_t


# ══════════════════════════════════════════════════════════════════
# 함수 2: Tweedie x_0 추정  (TOY의 model.posterior_mean() 대체)
#   x_t → 추정된 x_0
# ══════════════════════════════════════════════════════════════════
def tweedie_x0(x_t, t_idx):
    """
    Tweedie's formula: x0_hat = (x_t - sqrt(1-alpha_bar)*eps) / sqrt(alpha_bar)
    
    x_t   : 노이즈 낀 이미지 [B, C, H, W]
    t_idx : timestep 인덱스
    return: x_0 추정값
    """
    alpha_bar = alphas_cumprod[t_idx]
    t_tensor  = torch.tensor([t_idx], device=DEVICE)
    
    with torch.no_grad():
        eps_pred = model(x_t, t_tensor)   # UNet이 노이즈 예측
    
    x0_hat = (x_t - torch.sqrt(1 - alpha_bar) * eps_pred) / torch.sqrt(alpha_bar)
    return x0_hat


# ══════════════════════════════════════════════════════════════════
# 함수 3: DDIM step  (TOY의 ddim_step() 대체)
#   x_t → x_{t-1}
# ══════════════════════════════════════════════════════════════════
def ddim_step(x_t, x0_hat, t_idx):
    """
    x_t    : 현재 노이즈 상태
    x0_hat : 현재 x_0 추정값
    t_idx  : 현재 timestep
    return : x_{t-1}
    """
    alpha_bar_t    = alphas_cumprod[t_idx]
    alpha_bar_prev = alphas_cumprod[t_idx - 1]
    
    eps_pred = (x_t - torch.sqrt(alpha_bar_t) * x0_hat) / torch.sqrt(1 - alpha_bar_t)
    x_prev   = torch.sqrt(alpha_bar_prev) * x0_hat + torch.sqrt(1 - alpha_bar_prev) * eps_pred
    return x_prev


# ══════════════════════════════════════════════════════════════════
# 함수 4: Stanczuk SVD  (TOY의 model.U 대체)  ← 핵심!
#   score 벡터들을 K개 모아서 SVD → normal/tangent basis 추출
# ══════════════════════════════════════════════════════════════════
def get_stanczuk_basis(x_t, t_idx, K=200, noise_scale_factor=0.1):
    """
    Stanczuk et al. (ICML 2024) 알고리즘:
    
    핵심 아이디어:
      - score는 항상 normal 방향을 가리킴 (Theorem 4.1)
      - x_t를 K번 다르게 흔들면 → K개의 다른 normal 방향 score 획득
      - 이를 모아서 SVD → normal space의 basis 추출
    
    x_t              : 기준점 [B, C, H, W]
    t_idx            : timestep 인덱스
    K                : score 샘플 수 (클수록 정확, 느림)
    noise_scale_factor: 흔드는 크기 (sigma_t의 비율)
    
    return:
      normal_basis  : [D, codim]  (큰 SV 방향들 → normal space)
      tangent_basis : [D, k]      (작은 SV 방향들 → tangent space)
      singular_vals : singular values (spectral gap 확인용)
    """
    sigma_t   = torch.sqrt(1 - alphas_cumprod[t_idx]).item()
    noise_std = sigma_t * noise_scale_factor
    t_tensor  = torch.tensor([t_idx], device=DEVICE)
    D         = x_t.numel()  # 전체 차원 수 (예: 3*32*32 = 3072)

    score_vecs = []
    for _ in range(K):
        # Step 1: x_t를 작은 noise로 살짝 흔들기
        eps      = torch.randn_like(x_t)
        x_pert   = x_t + noise_std * eps

        # Step 2: 그 점에서 score 계산
        #         score = -eps_pred / sigma_t  (Tweedie 역관계)
        with torch.no_grad():
            eps_pred = model(x_pert, t_tensor)
        score = (-eps_pred / (sigma_t + 1e-8)).flatten()  # [D]
        score_vecs.append(score)

    # Step 3: [D, K] 행렬로 쌓기
    S = torch.stack(score_vecs, dim=1)  # [D, K]

    # Step 4: SVD
    #   - 큰 singular values → score들이 많이 분포 → normal space
    #   - 작은 singular values → score가 거의 없음 → tangent space
    U, singular_vals, _ = torch.linalg.svd(S, full_matrices=False)  # U: [D, K]

    # Step 5: Spectral gap으로 normal/tangent 경계 자동 탐지
    #   SV가 갑자기 뚝 떨어지는 지점 = normal과 tangent의 경계
    ratios  = singular_vals[:-1] / (singular_vals[1:] + 1e-12)
    gap_idx = torch.argmax(ratios).item() + 1  # gap 위치 = codimension

    normal_basis  = U[:, :gap_idx]   # [D, codim] ← 큰 SV들
    tangent_basis = U[:, gap_idx:]   # [D, k]     ← 작은 SV들

    return normal_basis, tangent_basis, singular_vals


# ══════════════════════════════════════════════════════════════════
# 함수 5: Error 분해  (TOY의 decompose_error(err, model.U) 대체)
#   error → normal 성분 / tangent 성분으로 분리
# ══════════════════════════════════════════════════════════════════
def decompose_error(err, normal_basis, tangent_basis):
    """
    err           : error 벡터 [B, C, H, W]
    normal_basis  : [D, codim]
    tangent_basis : [D, k]
    return: (normal_norm, tangent_norm)
    """
    e = err.flatten()  # [D]

    # normal 성분: normal_basis 방향으로 projection
    e_normal  = normal_basis @ (normal_basis.T @ e)
    # tangent 성분: tangent_basis 방향으로 projection
    e_tangent = tangent_basis @ (tangent_basis.T @ e)

    return e_normal.norm().item(), e_tangent.norm().item()


# ══════════════════════════════════════════════════════════════════
# 함수 6: Syndrome 계산  (TOY의 syndrome 계산과 동일한 로직)
#   syndrome = x0_curr - x0_lookahead
# ══════════════════════════════════════════════════════════════════
def compute_syndrome(x_t, t_idx):
    """
    syndrome: 두 번 denoising해서 얻은 x_0 추정치의 차이
    
    오차가 없으면 두 추정치는 같아야 함 → syndrome ≈ 0
    오차(특히 normal error)가 있으면 → syndrome 커짐
    """
    # Step 1: 현재 x_0 추정
    x0_curr = tweedie_x0(x_t, t_idx)

    # Step 2: DDIM으로 t → t-1
    x_t_prev = ddim_step(x_t, x0_curr, t_idx)

    # Step 3: t-1에서 x_0 재추정
    x0_look = tweedie_x0(x_t_prev, t_idx - 1)

    # Step 4: 두 추정치의 차이 = syndrome
    syndrome = x0_curr - x0_look
    return syndrome.norm().item()


print('모든 함수 정의 완료 ✓')
print(f'  - forward_diffuse()    : x_0 → x_t')
print(f'  - tweedie_x0()         : x_t → x_0 추정  (model.posterior_mean 대체)')
print(f'  - ddim_step()          : x_t → x_{{t-1}}')
print(f'  - get_stanczuk_basis() : Stanczuk SVD → normal/tangent basis  (model.U 대체)')
print(f'  - decompose_error()    : error → normal/tangent 분해')
print(f'  - compute_syndrome()   : syndrome 계산')

## Cell 4: Stanczuk SVD 동작 확인 (sanity check)

본 실험 전에 Stanczuk 분해가 제대로 동작하는지 먼저 확인한다.  
**Spectral gap이 뚜렷하게 보이면 정상.**

In [ ]:
print('Stanczuk SVD sanity check...')

# 테스트용 x_t 하나 생성
t_test_sanity = seq[50]   # 중간 정도 timestep
x_test = torch.randn(1, 3, 32, 32, device=DEVICE)

normal_b, tangent_b, sv = get_stanczuk_basis(x_test, t_test_sanity, K=100)

print(f'Timestep       : {t_test_sanity}')
print(f'Normal basis   : {normal_b.shape}  (차원: {normal_b.shape[1]})')
print(f'Tangent basis  : {tangent_b.shape} (차원: {tangent_b.shape[1]})')
print(f'Singular vals  : {sv[:10].cpu().numpy().round(3)} ...')

# Spectral gap 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sv.cpu().numpy(), 'o-', markersize=3)
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Singular Value')
axes[0].set_title(f'Singular Values at t={t_test_sanity}')
axes[0].grid(True, alpha=0.3)

ratios = (sv[:-1] / (sv[1:] + 1e-12)).cpu().numpy()
gap_idx = np.argmax(ratios) + 1
axes[1].plot(ratios, 'o-', markersize=3, color='orange')
axes[1].axvline(x=gap_idx-1, color='red', linestyle='--', label=f'Gap at {gap_idx}')
axes[1].set_xlabel('Index')
axes[1].set_ylabel('SV[i] / SV[i+1]')
axes[1].set_title('Spectral Gap (normal/tangent 경계)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'→ Spectral gap 위치: {gap_idx} (normal: {gap_idx}차원, tangent: {len(sv)-gap_idx}차원)')

## Cell 5: 실험 파라미터 설정

In [ ]:
# ── 실험 파라미터 ───────────────────────────────────────────────

# TOY 실험: [20, 50, 100, 150, 180] (T=200 기준)
# DDPM 실험: seq에서 비슷한 비율로 5개 선택
n_seq = len(seq)
TEST_TIMESTEPS = [
    seq[int(n_seq * 0.10)],   # ~초반
    seq[int(n_seq * 0.25)],   # ~1/4
    seq[int(n_seq * 0.50)],   # ~중간
    seq[int(n_seq * 0.75)],   # ~3/4
    seq[int(n_seq * 0.90)],   # ~후반
]
TEST_TIMESTEPS = sorted(set(TEST_TIMESTEPS))  # 중복 제거

ETA_VALUES = np.logspace(-2.5, -0.3, 10).tolist()  # perturbation 크기
N_PER      = 20    # timestep당 trial 수 (TOY는 80, DDPM은 비싸서 줄임)
K_SVD      = 200   # Stanczuk SVD용 score 샘플 수

print(f'테스트 timesteps : {TEST_TIMESTEPS}')
print(f'eta 범위         : {ETA_VALUES[0]:.4f} ~ {ETA_VALUES[-1]:.4f} ({len(ETA_VALUES)}개)')
print(f'trial 수 (N_PER) : {N_PER}')
print(f'SVD K            : {K_SVD}')
print(f'예상 UNet 호출 수 : {len(TEST_TIMESTEPS)} × (K_SVD + N_PER×eta×2) ≈ {len(TEST_TIMESTEPS)*(K_SVD+len(ETA_VALUES)*N_PER*2):,}회')

## Cell 6: 메인 실험 루프

TOY 실험의 `run_experiment_1()`과 동일한 구조.  
바뀐 부분:
- `model.U` → `get_stanczuk_basis()` 
- `model.posterior_mean()` → `tweedie_x0()`
- `model.project_normal/tangent()` → `decompose_error()`

In [ ]:
# results 딕셔너리 초기화 (TOY 코드와 동일한 구조)
results = {
    t: {'eta': [], 'syn_normal': [], 'syn_tangent': [],
        'e_normal': [], 'e_tangent': []}
    for t in TEST_TIMESTEPS
}

for t_test in TEST_TIMESTEPS:
    print(f'\n━━━ Timestep {t_test} ━━━')

    # ── Step A: Stanczuk SVD (timestep당 한 번만) ──────────────────
    # 비용 절감: 대표 x_t 하나에서 basis를 추출해서 재사용
    print(f'  Stanczuk SVD 계산중 (K={K_SVD})...', end=' ', flush=True)
    x_ref = torch.randn(1, 3, 32, 32, device=DEVICE)
    normal_basis, tangent_basis, sv = get_stanczuk_basis(
        x_ref, t_test, K=K_SVD
    )
    print(f'완료. normal {normal_basis.shape[1]}차원 / tangent {tangent_basis.shape[1]}차원')

    # ── Step B: eta 루프 ───────────────────────────────────────────
    for eta in tqdm(ETA_VALUES, desc=f'  eta loop'):
        syn_n_list, syn_t_list = [], []
        en_list,    et_list    = [], []

        for _ in range(N_PER):
            # x_t 생성 (ideal state)
            x_t_star = torch.randn(1, 3, 32, 32, device=DEVICE)

            for direction in ['normal', 'tangent']:
                # ── Step C: Perturbation 생성 ──────────────────────
                # random noise를 해당 방향으로 projection
                noise_raw = torch.randn(x_t_star.numel(), device=DEVICE)

                if direction == 'normal':
                    # normal basis 방향으로만 projection
                    noise = normal_basis @ (normal_basis.T @ noise_raw)
                else:
                    # tangent basis 방향으로만 projection
                    noise = tangent_basis @ (tangent_basis.T @ noise_raw)

                # 정규화 (TOY 코드와 동일)
                norm = noise.norm()
                if norm < 1e-12:
                    continue
                noise = noise / norm * (x_t_star.numel() ** 0.5)
                x_t_pert = x_t_star + eta * noise.view_as(x_t_star)

                # ── Step D: Error 분해 ─────────────────────────────
                # TOY: decompose_error(err, model.U)
                # DDPM: decompose_error(err, normal_basis, tangent_basis)
                err = x_t_pert - x_t_star
                en, et = decompose_error(err, normal_basis, tangent_basis)

                # ── Step E: Syndrome 계산 ──────────────────────────
                syn = compute_syndrome(x_t_pert, t_test)

                # 결과 수집
                if direction == 'normal':
                    syn_n_list.append(syn)
                    en_list.append(en)
                else:
                    syn_t_list.append(syn)
                    et_list.append(et)

        # 평균 저장
        results[t_test]['eta'].append(eta)
        results[t_test]['syn_normal'].append(np.mean(syn_n_list))
        results[t_test]['syn_tangent'].append(np.mean(syn_t_list))
        results[t_test]['e_normal'].append(np.mean(en_list))
        results[t_test]['e_tangent'].append(np.mean(et_list))

print('\n실험 완료 ✓')

## Cell 7: 결과 저장

In [ ]:
os.makedirs('./exp1_results', exist_ok=True)
torch.save(results, './exp1_results/exp1_ddpm_results.pt')
print('결과 저장 완료: ./exp1_results/exp1_ddpm_results.pt')

## Cell 8: 그래프

TOY 실험의 fig1과 동일한 형식으로 그린다.  

**Theorem 6.1이 DDPM에서도 성립한다면:**
- 그래프 (a): 기울기 가파름 → syndrome이 normal error에 민감
- 그래프 (b): 기울기 완만 → syndrome이 tangent error에 둔감  
- 그래프 (c): κ/β >> 1 → 관측가능성 비율이 크다

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
COLORS = ['#C83232','#28A050','#005AA0','#E07814','#6A5ACD']

# ── 그래프 (a): Syndrome vs Normal Error ─────────────────────────
ax = axes[0]
for i, t in enumerate(TEST_TIMESTEPS):
    r = results[t]
    ax.plot(r['e_normal'], r['syn_normal'], 'o-',
            color=COLORS[i], markersize=3, label=f't={t}', alpha=0.8)
ax.set_xlabel(r'$\|e_t^\perp\|$ (normal error)')
ax.set_ylabel(r'$\|s_t\|$ (syndrome)')
ax.set_title('(a) Syndrome vs Normal Error\n→ 기울기 가파를수록 정상')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── 그래프 (b): Syndrome vs Tangent Error ────────────────────────
ax = axes[1]
for i, t in enumerate(TEST_TIMESTEPS):
    r = results[t]
    ax.plot(r['e_tangent'], r['syn_tangent'], 's-',
            color=COLORS[i], markersize=3, label=f't={t}', alpha=0.8)
ax.set_xlabel(r'$\|e_t^\parallel\|$ (tangent error)')
ax.set_ylabel(r'$\|s_t\|$ (syndrome)')
ax.set_title('(b) Syndrome vs Tangent Error\n→ 기울기 완만할수록 정상')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── 그래프 (c): κ/β 비율 ─────────────────────────────────────────
ax = axes[2]
ratios = []
for t in TEST_TIMESTEPS:
    r  = results[t]
    en = np.array(r['e_normal'])
    sn = np.array(r['syn_normal'])
    et = np.array(r['e_tangent'])
    st = np.array(r['syn_tangent'])
    mask_n = en > 1e-10
    mask_t = et > 1e-10
    if mask_n.sum() > 2 and mask_t.sum() > 2:
        kappa = np.polyfit(en[mask_n], sn[mask_n], 1)[0]
        beta  = max(np.polyfit(et[mask_t], st[mask_t], 1)[0], 1e-10)
        ratios.append(kappa / beta)
    else:
        ratios.append(0.0)

bars = ax.bar(range(len(TEST_TIMESTEPS)), ratios,
              color='#E07814', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(TEST_TIMESTEPS)))
ax.set_xticklabels([f't={t}' for t in TEST_TIMESTEPS])
ax.set_ylabel(r'$\kappa_t / \beta_t$')
ax.set_title(r'(c) Observability ratio $\kappa/\beta$' + '\n→ 1보다 크면 Theorem 6.1 성립')
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7, label='기준선 (1.0)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 값 표시
for bar, val in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Experiment 1: Syndrome Observability (DDPM 버전)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('./exp1_results/fig1_observability_ddpm.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n κ/β 비율: {[f"{r:.2f}" for r in ratios]}')
print(f' 평균 κ/β: {np.mean(ratios):.2f}')
print(f' → 모든 비율이 1보다 크면 Theorem 6.1이 DDPM에서도 성립!')

## Cell 9: TOY vs DDPM 비교

TOY 실험 결과가 있다면 나란히 비교한다.

In [ ]:
# TOY 결과 파일이 있으면 비교, 없으면 DDPM 결과만 출력
TOY_RESULT_PATH = './results/exp1_toy_results.pt'  # TOY 결과 경로 (있으면 수정)

print('=== Experiment 1 최종 요약 ===')
print(f'\n[DDPM 결과]')
print(f'{"Timestep":>10} {"κ/β":>8} {"판정":>8}')
print('-' * 32)
for t, ratio in zip(TEST_TIMESTEPS, ratios):
    status = '✓ 성립' if ratio > 1.0 else '✗ 불성립'
    print(f'{t:>10} {ratio:>8.2f} {status:>8}')

print(f'\n결론: κ/β > 1인 timestep = {sum(1 for r in ratios if r > 1)}/{len(ratios)}')

if os.path.exists(TOY_RESULT_PATH):
    print('\n[TOY 결과와 비교]')
    toy_results = torch.load(TOY_RESULT_PATH)
    # TOY 결과 비교 로직 (TOY 결과 구조에 맞게 수정 필요)
    print('TOY 결과 로드됨. 비교 그래프는 별도로 그릴 수 있음.')
else:
    print(f'\nTOY 결과 파일이 없어서 비교 생략: {TOY_RESULT_PATH}')